In [26]:
import os
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torchvision import transforms
from torchvision.utils import save_image

In [27]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# I. Download dataset

In [28]:
# MNIST Dataset
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [29]:
sample_dir = "results"
if not os.path.exists(sample_dir):
    os.makedirs(sample_dir)

# II. Define model

## 1. Model

In [30]:
class VAE(nn.Module):
    def __init__(self, image_size, hidden_dim, latent_dim):
        super(VAE, self).__init__()

        self.fc1 = nn.Linear(image_size, hidden_dim)
        self.fc2_mean = nn.Linear(hidden_dim, latent_dim)
        self.fc2_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, image_size)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc2_mean(h)
        logvar = self.fc2_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(logvar/2)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc3(z))
        out = torch.sigmoid(self.fc4(h))
        return out

    def forward(self, x):
        # x: [batch_size, 1, 28, 28] -> [batch_size, 784]
        mu, logvar = self.encode(x.view(-1, image_size))
        # mu, logvar: [batch_size, latent_dim]

        z = self.reparameterize(mu, logvar)
        # z: [batch_size, latent_dim]

        reconstructed = self.decode(z)
        # reconstructed: [batch_size, image_size]

        return reconstructed, mu, logvar

In [38]:
batch_size = 128
image_size = 784
hidden_dim = 400
latent_dim = 10
epochs = 1

In [32]:
model = VAE(image_size=image_size, hidden_dim=hidden_dim, latent_dim=latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 2. Loss function

\begin{align}
 \textrm{Loss} &= -E\left [ \textrm{log } \textrm{P}\left ( X|z \right ) \right ] + D_{KL}\left [\: N\left ( \mu (X), \sum (X) \right | N\left ( 0,1 \right ) ) \right ], \quad \\
                &\textrm{where } D_{KL}\left [\: N\left ( \mu (X), \sum (X) \right | N\left ( 0,1 \right ) ) \right ]
                = \frac{1}{2} \sum \left [ \textrm{exp}\left ( \sum (X) \right ) + \mu ^2\left ( X \right ) - 1 - \sum \left ( X \right ) \right ]
\end{align}

In [33]:
def loss_function(reconstructed_image, original_image, mu, log_var):
    bce = F.binary_cross_entropy(reconstructed_image, original_image.view(-1, 784), reduction = "sum")
    kld = 0.5 * torch.sum(log_var.exp() + mu.pow(2) - 1 - log_var)
    return bce + kld

## 3. Training function

In [34]:
def train(epoch):
    model.train()
    train_loss = 0

    for i, (images, _) in enumerate(train_loader):
        images = images.to(device)
        reconstructed, mu, logvar = model(images) # require_grad = True already !
        loss = loss_function(reconstructed, images, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        if i % 100 == 0:
            print("Train Epoch: {} [{}/{}]\tLoss: {:.3f}".format(epoch, i, len(train_loader), loss.item()/len(images)))

    print("=====> Epoch {} with average loss: {:.3f}".format(epoch, train_loss/len(train_loader.dataset)))

## 4. Testing function

In [35]:
def test(epoch):
    model.eval()
    test_loss = 0

    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(test_loader):
            images = images.to(device)
            reconstructed, mu, logvar = model(images)
            test_loss += loss_function(reconstructed, images, mu, logvar).item()

            if batch_idx == 0:
                comparison = torch.cat([images[:5], reconstructed.view(batch_size, 1, 28, 28)[:5]])
                save_image(comparison.cpu(), "results/reconstruction_" + str(epoch) + ".png", nrow=5)

    print("=====> Epoch {} with average loss: {:.3f}".format(epoch, test_loss/len(test_loader.dataset)))

# III. Train & Evaluate Model

In [39]:
for epoch in range(1, epochs+1):
    train(epoch)
    test(epoch)

    with torch.no_grad():
        sample = torch.randn(64,20).to(device)
        generate = model.decode(sample).cpu()
        save_image(generate.view(64, 1, 28, 28), "results/sample_" + str(epoch)+ ".png")

Train Epoch: 1 [0/469]	Loss: 111.926
Train Epoch: 1 [100/469]	Loss: 113.684
Train Epoch: 1 [200/469]	Loss: 112.285
Train Epoch: 1 [300/469]	Loss: 114.163
Train Epoch: 1 [400/469]	Loss: 115.024
=====> Epoch 1 with average loss: 114.084
=====> Epoch 1 with average loss: 111.281


**Result of model:**

<div align="center">
  <img src="https://drive.google.com/uc?export=view&id=1BFWafSNY-bznRQZFBMfdlXuJIfKdJKuO" width="300"/>
</div>



